This code has been prouced as a teaching resource for the UKSA space software, data and AI course run by the Space South Central Universities.

Contributors to this code includes: O. Umeh; B. Canning; A. Tolley

---

### Learning Outcome
<div class="alert alert-block alert-info"> 
<b>NOTE</b> In this notebook we will learn how to ingest our prepared data and fine-tune a YOLO Computer Vision model.
</div>

---

### Standard Package Imports

In [ ]:
import os
import shutil
import numpy as np
import matplotlib.pyplot as plt
import geojson
import geopandas as gpd
from PIL import Image

from ultralytics import YOLO
import yaml
import json
import glob
import cv2

In [ ]:
import torch
if torch.cuda.is_available():
    device = 'cuda'
else:
    device = 'cpu'

print(f"Using device: {device}")

---

### Preamble

In Casestudy Part 2 we picked Palu, Indonesia as a region to investigate the identification of earthquake damaged buildings.

In that notebook we looked at using OSM data and downloading corresponding Sentinel-2 satellite imagery of Palu before creating an ML ready dataset to finetune a computer vision model.

**Our Aims**:
- Use the dataset to train a Computer Vision (CV) Machine Learning (ML) algorithm to detect buildings in an unmapped location

---

### Ingest data made in Part 2

The dataset we made in Part 2 was in the **Common Objects in Context** (COCO) format, a widely-used annotation format in computer vision with an accompanying COCO dataset which is used as a benchmark for evaluating computer vision models which attempt to do things like object detection.

We are going to be using the **You Only Look Once** algorithm for our building detection which requires a slightly different data format, therefore, we need to do the conversion.

Converting between formats is a common task in Machine Learning, a great way to do this easily is by asking an LLM (like ChatGPT) for a script that can do the conversion.

In [ ]:
# Directories
tile_dir = './casestudy_data/pre_earthquake_tiles/'
mask_dir = './casestudy_data/masks/'
annotations_dir = './casestudy_data/output_annotations/'
annotations_file = annotations_dir + 'annotations.json'

output_dir = './casestudy_data/yolo_dataset/'
images_dir = os.path.join(output_dir, 'images')
labels_dir = os.path.join(output_dir, 'labels')

os.makedirs(images_dir, exist_ok=True)
os.makedirs(labels_dir, exist_ok=True)

# Load annotation JSON
with open(annotations_file, 'r') as f:
    data = json.load(f)

# Create YOLO label files and copy images
for item in data:
    image_path = os.path.join(tile_dir, item['tile_path'])
    boxes = item['boxes']
    labels = item['labels']

    # Get image size
    with Image.open(image_path) as img:
        img_width, img_height = img.size

    # Create YOLO label file
    txt_filename = os.path.splitext(os.path.basename(image_path))[0] + '.txt'
    txt_path = os.path.join(labels_dir, txt_filename)

    with open(txt_path, 'w') as f:
        for box, label in zip(boxes, labels):
            x_min, y_min, x_max, y_max = box
            x_center = ((x_min + x_max) / 2) / img_width
            y_center = ((y_min + y_max) / 2) / img_height
            width = (x_max - x_min) / img_width
            height = (y_max - y_min) / img_height

            f.write(f"{label - 1} {x_center:.6f} {y_center:.6f} {width:.6f} {height:.6f}\n")

    # Copy image
    shutil.copy(image_path, os.path.join(images_dir, os.path.basename(image_path)))

print(f"✅ Dataset created in {output_dir}")

# Create train.txt
with open(os.path.join(output_dir, 'train.txt'), 'w') as f:
    for item in data:
        f.write(f"./images/{item['tile_path']}\n")

We need to create a `yaml` file while describes the split of our files between testing, training and validation

In [ ]:
# Create data.yaml
data_yaml = {
    'train': os.path.abspath(images_dir),
    'val': os.path.abspath(images_dir),
    'nc': 1,
    'names': ['building']
}

with open(os.path.join(output_dir, 'data.yaml'), 'w') as f:
    yaml.dump(data_yaml, f)

---

### Fine Tune YOLO

Now we have converted our data and loaded it in, we can move on to fine tuning the YOLO model with our data.

In [47]:
# YOLOv8 training
yaml_path = os.path.join(output_dir, 'data.yaml')
epochs = 10
img_size = 32 # Suggested YOLO default
batch_size = 4

model = YOLO('yolov8n.pt')

results = model.train(
    data=yaml_path,
    epochs=epochs,
    imgsz=img_size,
    batch=batch_size,
    name='yolov8_fine_tuned',
    device='cpu'
)

# Optional: Validate
val_results = model.val(data=yaml_path, device='mps')
print(f"📊 Validation results: {val_results}")


                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 633/633 [29:16<00:00,  2.78s/it]


                   all       2532     166550    0.00659    0.00459     0.0034    0.00047
Speed: 0.3ms preprocess, 3.9ms inference, 0.0ms loss, 11.8ms postprocess per image
Results saved to /Users/arthur/Documents/coding/cpd-notebooks/runs/detect/yolov8_fine_tuned82
📊 Validation results: ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x44a940360>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027

---

### Visualise Results

We now have a finetuned YOLOv8 model, saved as `model`.

We can use this model on our testing area to see how well it has worked

#### We're going to define some visualisation functions:

Calculate the intersection of two boxes:

In [48]:
def iou(box1, box2):
    # Calculate intersection area
    x1 = max(box1[0], box2[0])
    y1 = max(box1[1], box2[1])
    x2 = min(box1[2], box2[2])
    y2 = min(box1[3], box2[3])

    intersection_area = max(0, x2 - x1) * max(0, y2 - y1)

    # Calculate areas
    box1_area = (box1[2] - box1[0]) * (box1[3] - box1[1])
    box2_area = (box2[2] - box2[0]) * (box2[3] - box2[1])

    # Calculate IoU
    return intersection_area / float(box1_area + box2_area - intersection_area)

Load an image and run the fine-tuned model on it 

In [49]:
def process_and_visualize(model_path, image_path, confidence_threshold=0.35, iou_threshold=0.45):
    # Load fine-tuned model
    fine_tuned_model = YOLO(model_path)

    # Load the image
    img = cv2.imread(image_path)
    #print("inout image shape", img.shape[:2])

    # Run inference
    results = fine_tuned_model(img)[0]

    # List to keep track of boxes to draw
    boxes_to_draw = []

    for box in results.boxes:
        # Extract coordinates, class, and confidence score
        x_min, y_min, x_max, y_max = map(int, box.xyxy[0])  # Coordinates of the bounding box
        conf = box.conf.item()  # Confidence score
        cls = int(box.cls.item())  # Class index

        # Only consider boxes with confidence above the threshold
        if conf >= confidence_threshold:
            current_box = (x_min, y_min, x_max, y_max)
            is_best = True

            # Check for substantial overlap with already accepted boxes
            for existing_box in boxes_to_draw:
                if iou(current_box, existing_box['bbox']) > iou_threshold:
                    is_best = False
                    break

            if is_best:
                boxes_to_draw.append({
                    'bbox': current_box,
                    'conf': conf,
                    'label': fine_tuned_model.names[cls]
                })

    # Visualize the best detections (including all non-overlapping)
    for det in boxes_to_draw:
        x_min, y_min, x_max, y_max = det['bbox']
        label = det['label']
        conf = det['conf']

        # Draw the bounding box
        cv2.rectangle(img, (x_min, y_min), (x_max, y_max), (0, 255, 0), 1)

        # Draw the label and confidence score
        #cv2.putText(img, f'{label}: {conf:.2f}', (x_min, y_min - 10),
      #              cv2.FONT_HERSHEY_SIMPLEX, 0.1, (255, 0, 0), 1)

    # Convert BGR (OpenCV) image to RGB for visualization
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

    # Display the image with bounding boxes
    plt.imshow(img_rgb)
    plt.show()

Run the model

In [51]:
dataset = os.path.join(output_dir, 'images')

# Use the weights generated by the YOLO fine-tuning cell above.
model_weights = os.path.join('runs', 'detect', 'yolov8_fine_tuned', 'weights', 'best.pt')

# Process and visualize the results.
image_files = glob.glob(os.path.join(dataset, '*.png'))
for image_file in image_files[:3]:
    print(f"Processing {image_file}...")
    process_and_visualize(model_weights, image_file, confidence_threshold=0.35, iou_threshold=0.45)


Processing ./casestudy_data/yolo_dataset/images/tile_736_1184.png...

0: 32x32 (no detections), 5.4ms
Speed: 0.3ms preprocess, 5.4ms inference, 0.3ms postprocess per image at shape (1, 3, 32, 32)


<Figure size 640x480 with 1 Axes>

Processing ./casestudy_data/yolo_dataset/images/tile_820_1040.png...

0: 32x32 (no detections), 4.3ms
Speed: 0.2ms preprocess, 4.3ms inference, 0.2ms postprocess per image at shape (1, 3, 32, 32)


<Figure size 640x480 with 1 Axes>

Processing ./casestudy_data/yolo_dataset/images/tile_320_1060.png...

0: 32x32 (no detections), 4.8ms
Speed: 0.3ms preprocess, 4.8ms inference, 0.2ms postprocess per image at shape (1, 3, 32, 32)


<Figure size 640x480 with 1 Axes>

---